In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
from json import load as load_json

In [ ]:
_Delay = 2.5

with open("config.json", "r") as f:
        headers = load_json(f)
headers

In [ ]:
awards = [
        {
                "ID": 0,
                "Award Short Name": "MVP",
                "Trophy Name": "Michael Jordan Trophy",
                "Description": "Most Valuable Player",
        },
        {
                "ID": 1,
                "Award Short Name": "All Star",
                "Trophy Name": "Kobe Bryant Trophy",
                "Description": "All-Star Game Most Valuable Player",
        },
        {
                "ID": 2,
                "Award Short Name": "Finals MVP",
                "Trophy Name": "Bill Russell Trophy",
                "Description": "Finals Most Valuable Player",
        },
        {
                "ID": 3,
                "Award Short Name": "ROY",
                "Trophy Name": "Wilt Chamberlain Trophy",
                "Description": "Rookie of the Year",
        },
        {
                "ID": 4,
                "Award Short Name": "DPOY",
                "Trophy Name": "Hakeem Olajuwon Trophy",
                "Description": "Defensive Player of the Year",
        }
]

pd.DataFrame(awards).to_csv("../../data/raw/awards.csv", index = False)

In [ ]:
with open("../../data/raw/awards_url.txt", "r") as f:
    awards_url = f.read().splitlines()
awards_url[:5]

columns = [
        "ID",
        "Season",
        "Player Name",
        "Award ID",
]
try:
    award_season_data = pd.read_csv("../../data/raw/award_season_data.csv")
except:
    award_season_data = pd.DataFrame(columns = columns)
# award_season_data = pd.DataFrame(columns = columns)
award_season_data.head()

In [ ]:
def get_value(row, data_stat: str):
        try:
                return row.find(attrs={"data-stat": data_stat}).a.text.strip()
        except:
                return None

def get_award_info(url: str, headers: dict, columns: list, start_id: int, start_selector: str, award_id: int):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        df = pd.DataFrame(columns = columns)
        
        while True:
                try:
                        info_tag = soup.select(start_selector)
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        
        lst = info_tag[0].select("tr")
        info = [row for row in lst if row.get("class") == None or "sr_added_headers" not in row.get("class")]
        
        for row in info:
                if (get_value(row, "lg_id") != "NBA"):
                        continue
                
                result = {}
                # ID
                result[columns[0]] = len(df) + start_id
                # Season
                result[columns[1]] = get_value(row, "season")
                # Player Name
                result[columns[2]] = get_value(row, "player")
                # Award ID
                result[columns[3]] = award_id
                
                df.loc[len(df)] = result
        
        time.sleep(_Delay)
        return df

In [ ]:
start_selectors = [
        "#mvp_NBA > tbody",
        "#all_star_mvp_NBA > tbody",
        "#finals_mvp_NBA > tbody",
        "#roy_NBA > tbody",
        "#dpoy_NBA > tbody"
]

award_season_data = pd.DataFrame(columns = columns)
row_id = 0

In [ ]:
for i in tqdm(range(len(awards_url))):
        result = get_award_info(
                url=awards_url[i],
                headers=headers,
                columns=columns,
                start_id= row_id,
                start_selector=start_selectors[i],
                award_id = i
        )
        
        row_id += len(result)
        
        award_season_data = pd.concat([award_season_data, result], ignore_index = True)
        award_season_data.to_csv("../../data/raw/award_season_data.csv", index = False)